# Machine Learning Project

**End-to-End Supervised Learning Pipeline for Google Colab**

This notebook implements a complete ML workflow: data loading → EDA → preprocessing → ensemble model training → evaluation → automated report generation.

**How to use:**
1. Run all cells sequentially (Runtime → Run all).
2. Upload your dataset when prompted (CSV, XLSX, or JSON).
3. Optionally override `TARGET_COLUMN` below if the target is not the last column.
4. Paste errors, metrics, or screenshots back for iterative improvements.


In [ ]:
# Install required packages (Colab-compatible)
!pip install -q imbalanced-learn openpyxl

# Enable inline plotting in Colab
%matplotlib inline


In [ ]:
"""
Global configuration and imports.
random_state=42 is used everywhere for reproducibility.
"""
import warnings
warnings.filterwarnings("ignore")

import io
import json
import os
import re
import textwrap
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import files
from IPython.display import display
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.ensemble import (
    BaggingClassifier,
    BaggingRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
    StackingClassifier,
    StackingRegressor,
    VotingClassifier,
    VotingRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from scipy.stats import randint, uniform

# ---------------------------------------------------------------------------
# USER CONFIGURATION — edit these if needed
# ---------------------------------------------------------------------------
TARGET_COLUMN = "toxic"       # None = auto-use last column; use quotes: "toxic", "price", etc.
# Other Jigsaw toxic labels — auto-removed from FEATURES when predicting one of them
SIBLING_LABEL_COLUMNS = ["severe_toxic", "obscene", "threat", "insult", "identity_hate"]
RANDOM_STATE = 42
TEST_SIZE = 0.2
MISSING_DROP_THRESHOLD = 0.50   # drop columns with >50% missing
CORR_THRESHOLD = 0.95           # drop one of highly correlated feature pairs
CV_FOLDS = 5
N_ITER_SEARCH = 20
PAIRPLOT_MAX_FEATURES = 6       # limit pairplot size for performance
MAX_TRAINING_ROWS = 50000         # subsample for Colab if dataset is larger; set None to use all rows

# Plot styling
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# Containers populated during pipeline execution (used in final report)
PIPELINE_LOG: Dict[str, Any] = {
    "dataset_summary": {},
    "eda_findings": [],
    "preprocessing": [],
    "hyperparameters": {},
    "metrics": {},
    "best_model": "",
    "conclusions": [],
}

print("Configuration loaded. RANDOM_STATE =", RANDOM_STATE)


## Step 1: Dataset Loading

Upload a dataset file. Supported formats:
- **CSV** (`.csv`)
- **Excel** (`.xlsx`, `.xls`)
- **JSON** (`.json`)

The target column defaults to the **last column**. Set `TARGET_COLUMN` in the configuration cell to override.


In [ ]:
def upload_dataset() -> Tuple[str, bytes]:
    """Prompt user to upload a file in Colab and return (filename, raw bytes)."""
    print("Please upload your dataset file (CSV, XLSX, or JSON)...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded. Please re-run this cell and upload a dataset.")
    filename = next(iter(uploaded.keys()))
    return filename, uploaded[filename]


def load_dataframe(filename: str, raw_bytes: bytes) -> pd.DataFrame:
    """Load uploaded bytes into a pandas DataFrame based on file extension."""
    ext = os.path.splitext(filename.lower())[1]

    if ext == ".csv":
        # Try common encodings for robust CSV loading
        for encoding in ("utf-8", "latin-1", "cp1252"):
            try:
                df = pd.read_csv(io.BytesIO(raw_bytes), encoding=encoding, low_memory=False)
                print(f"Loaded CSV with encoding='{encoding}'")
                return df
            except UnicodeDecodeError:
                continue
        raise ValueError("Could not decode CSV file. Try saving as UTF-8.")

    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(io.BytesIO(raw_bytes))
        print("Loaded Excel file")
        return df

    if ext == ".json":
        try:
            df = pd.read_json(io.BytesIO(raw_bytes))
        except ValueError:
            df = pd.json_normalize(json.loads(raw_bytes.decode("utf-8")))
        print("Loaded JSON file")
        return df

    raise ValueError(f"Unsupported file extension: {ext}. Use CSV, XLSX, or JSON.")


def detect_target_column(df: pd.DataFrame, override: Optional[str]) -> str:
    """Return target column name; default is last column."""
    if override is not None:
        if override not in df.columns:
            raise ValueError(f"TARGET_COLUMN '{override}' not found. Columns: {list(df.columns)}")
        return override
    return df.columns[-1]


def detect_problem_type(y: pd.Series) -> str:
    """
    Classification if target has <= 20 unique values, else Regression.
    Coerce to numeric when possible for detection.
    """
    y_clean = pd.to_numeric(y, errors="coerce")
    if y_clean.notna().mean() > 0.95:
        n_unique = y_clean.nunique(dropna=True)
    else:
        n_unique = y.nunique(dropna=True)

    problem = "classification" if n_unique <= 20 else "regression"
    print(f"Detected problem type: {problem.upper()} (unique target values: {n_unique})")
    return problem


def coerce_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Attempt to convert object columns to numeric where possible."""
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == "object":
            converted = pd.to_numeric(out[col], errors="coerce")
            if converted.notna().mean() > 0.8:
                out[col] = converted
    return out


def drop_unusable_columns(df: pd.DataFrame, target_col: str) -> pd.DataFrame:
    """
    Drop ID-like and very high-cardinality text columns that break encoding.
    Keeps target column intact.
    """
    cols_to_drop = []
    n_rows = len(df)

    for col in df.columns:
        if col == target_col:
            continue
        series = df[col]
        nunique = series.nunique(dropna=True)

        # Drop constant columns
        if nunique <= 1:
            cols_to_drop.append(col)
            continue

        # Drop likely ID columns
        if re.search(r"(^id$|_id$|^index$)", col, re.IGNORECASE) and nunique > 0.9 * n_rows:
            cols_to_drop.append(col)
            continue

        # Drop free-text / ultra-high cardinality object columns
        if series.dtype == "object" and nunique > max(100, 0.5 * n_rows):
            cols_to_drop.append(col)

    if cols_to_drop:
        print(f"Dropping unusable/high-cardinality columns: {cols_to_drop}")
        PIPELINE_LOG["preprocessing"].append(
            f"Dropped unusable columns before modeling: {cols_to_drop}"
        )
        df = df.drop(columns=cols_to_drop)

    return df


In [ ]:
# --- Execute Step 1: Load dataset ---
filename, raw_bytes = upload_dataset()
df_raw = load_dataframe(filename, raw_bytes)
df_raw = coerce_numeric_columns(df_raw)

target_col = detect_target_column(df_raw, TARGET_COLUMN)
problem_type = detect_problem_type(df_raw[target_col])

PIPELINE_LOG["dataset_summary"] = {
    "filename": filename,
    "rows": len(df_raw),
    "columns": len(df_raw.columns),
    "target_column": target_col,
    "problem_type": problem_type,
    "column_names": list(df_raw.columns),
}

print("\n" + "=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print(f"Shape: {df_raw.shape}")
print(f"Target column: {target_col}")
print(f"Problem type: {problem_type}")
df_raw.head()


## Step 2: Exploratory Data Analysis (EDA)

Comprehensive exploration of the dataset before preprocessing. All plots display inline in Colab.


In [ ]:
def eda_dataset_overview(df: pd.DataFrame) -> None:
    """Print shape, dtypes, memory usage, and preview."""
    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)
    print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    print("\nData types:")
    print(df.dtypes)
    print("\nFirst 5 rows:")
    display(df.head())
    print("\nLast 5 rows:")
    display(df.tail())


def eda_descriptive_stats(df: pd.DataFrame) -> None:
    """Show descriptive statistics for numeric and categorical columns."""
    print("=" * 60)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 60)
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    if num_cols:
        print("\nNumeric features:")
        display(df[num_cols].describe().T)
    else:
        print("No numeric columns found.")

    if cat_cols:
        print("\nCategorical features (top values):")
        for col in cat_cols[:10]:
            print(f"\n--- {col} ---")
            display(df[col].value_counts(dropna=False).head(10))


def eda_missing_values(df: pd.DataFrame) -> None:
    """Visualize missing value counts and percentages."""
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
    missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)

    print("=" * 60)
    print("MISSING VALUES")
    print("=" * 60)
    if missing_df.empty:
        print("No missing values detected.")
        PIPELINE_LOG["eda_findings"].append("No missing values in raw data.")
        return

    display(missing_df)
    PIPELINE_LOG["eda_findings"].append(
        f"Missing values found in {len(missing_df)} column(s). Max missing: {missing_df['Missing %'].max():.1f}%"
    )

    plt.figure(figsize=(10, max(4, len(missing_df) * 0.35)))
    sns.barplot(x=missing_df["Missing %"], y=missing_df.index, palette="Reds_r")
    plt.title("Missing Values by Column (%)")
    plt.xlabel("Missing %")
    plt.tight_layout()
    plt.show()


def eda_duplicates(df: pd.DataFrame) -> int:
    """Report and visualize duplicate rows."""
    n_dup = df.duplicated().sum()
    print("=" * 60)
    print("DUPLICATE ROWS")
    print("=" * 60)
    print(f"Duplicate rows: {n_dup:,} ({n_dup / len(df) * 100:.2f}%)")
    PIPELINE_LOG["eda_findings"].append(f"Duplicate rows: {n_dup} ({n_dup / len(df) * 100:.2f}%)")

    plt.figure(figsize=(5, 4))
    dup_labels = pd.Series(np.where(df.duplicated(), "Duplicate", "Unique"))
    sns.countplot(x=dup_labels, order=["Unique", "Duplicate"])
    plt.title("Duplicate vs Unique Rows")
    plt.tight_layout()
    plt.show()
    return n_dup


def eda_target_distribution(df: pd.DataFrame, target: str, problem: str) -> None:
    """Plot target distribution for classification or regression."""
    print("=" * 60)
    print("TARGET DISTRIBUTION")
    print("=" * 60)

    y = df[target]
    if problem == "classification":
        counts = y.value_counts()
        display(counts.to_frame("count"))
        plt.figure(figsize=(8, 5))
        ax = sns.countplot(x=y.astype(str), order=counts.index.astype(str))
        ax.set_title(f"Class Distribution — {target}")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
        imbalance_ratio = counts.min() / counts.max() if counts.max() > 0 else 1
        PIPELINE_LOG["eda_findings"].append(
            f"Target classes: {len(counts)}. Imbalance ratio (min/max): {imbalance_ratio:.4f}"
        )
    else:
        display(y.describe())
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        sns.histplot(y, kde=True, bins=30)
        plt.title(f"Target Histogram — {target}")
        plt.subplot(1, 2, 2)
        sns.boxplot(x=y)
        plt.title("Target Boxplot")
        plt.tight_layout()
        plt.show()
        PIPELINE_LOG["eda_findings"].append(
            f"Target range: [{y.min()}, {y.max()}], mean={y.mean():.4f}, std={y.std():.4f}"
        )


def eda_correlation_heatmap(df: pd.DataFrame) -> None:
    """Correlation heatmap for numeric features."""
    num_df = df.select_dtypes(include=[np.number])
    if num_df.shape[1] < 2:
        print("Not enough numeric columns for correlation heatmap.")
        return

    print("=" * 60)
    print("CORRELATION HEATMAP")
    print("=" * 60)
    corr = num_df.corr(numeric_only=True)
    plt.figure(figsize=(min(14, 0.5 + num_df.shape[1]), min(12, 0.5 + num_df.shape[1])))
    sns.heatmap(corr, annot=num_df.shape[1] <= 15, fmt=".2f", cmap="coolwarm", center=0, square=True)
    plt.title("Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()

    # Log highly correlated pairs
    high_corr = []
    cols = corr.columns
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            if abs(corr.iloc[i, j]) > CORR_THRESHOLD:
                high_corr.append((cols[i], cols[j], corr.iloc[i, j]))
    if high_corr:
        PIPELINE_LOG["eda_findings"].append(f"Highly correlated pairs (|r|>{CORR_THRESHOLD}): {len(high_corr)}")


def eda_feature_histograms(df: pd.DataFrame, target: str) -> None:
    """Histograms for numeric features (excluding target)."""
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
    if not num_cols:
        return
    print("=" * 60)
    print("FEATURE HISTOGRAMS")
    print("=" * 60)
    n = min(len(num_cols), 12)
    cols = num_cols[:n]
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.histplot(df[col].dropna(), kde=True, ax=ax)
        ax.set_title(col)
    for ax in axes[len(cols):]:
        ax.axis("off")
    plt.suptitle("Numeric Feature Distributions", y=1.02)
    plt.tight_layout()
    plt.show()


def eda_boxplots(df: pd.DataFrame, target: str) -> None:
    """Boxplots for numeric features to visualize spread and outliers."""
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
    if not num_cols:
        return
    print("=" * 60)
    print("BOXPLOTS")
    print("=" * 60)
    n = min(len(num_cols), 12)
    cols = num_cols[:n]
    plt.figure(figsize=(14, max(4, n * 0.4)))
    melted = df[cols].melt(var_name="Feature", value_name="Value")
    sns.boxplot(data=melted, x="Value", y="Feature", orient="h")
    plt.title("Feature Boxplots (IQR-based outlier visualization)")
    plt.tight_layout()
    plt.show()


def eda_pairplot(df: pd.DataFrame, target: str, problem: str) -> None:
    """Pairplot for a subset of numeric features (performance-limited)."""
    num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target]
    if len(num_cols) < 2:
        print("Not enough numeric features for pairplot.")
        return
    cols = num_cols[:PAIRPLOT_MAX_FEATURES]
    plot_df = df[cols + [target]].copy()
    if problem == "classification" and plot_df[target].nunique() <= 20:
        plot_df[target] = plot_df[target].astype(str)
    print("=" * 60)
    print(f"PAIRPLOT (top {len(cols)} numeric features)")
    print("=" * 60)
    sns.pairplot(plot_df, hue=target if len(cols) <= 5 else None, diag_kind="hist", corner=True)
    plt.suptitle("Pairplot", y=1.02)
    plt.show()


def eda_class_imbalance(df: pd.DataFrame, target: str, problem: str) -> None:
    """Class imbalance analysis (classification only)."""
    if problem != "classification":
        print("Class imbalance analysis applies to classification tasks only.")
        return
    print("=" * 60)
    print("CLASS IMBALANCE ANALYSIS")
    print("=" * 60)
    counts = df[target].value_counts()
    proportions = (counts / len(df) * 100).round(2)
    imbalance_df = pd.DataFrame({"Count": counts, "Percentage": proportions})
    display(imbalance_df)

    plt.figure(figsize=(8, 5))
    plt.pie(counts, labels=counts.index.astype(str), autopct="%1.1f%%", startangle=90)
    plt.title("Class Proportion Pie Chart")
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Execute Step 2: EDA ---
df_eda = df_raw.copy()
eda_dataset_overview(df_eda)
eda_descriptive_stats(df_eda)
eda_missing_values(df_eda)
n_duplicates_eda = eda_duplicates(df_eda)
eda_target_distribution(df_eda, target_col, problem_type)
eda_correlation_heatmap(df_eda)
eda_feature_histograms(df_eda, target_col)
eda_boxplots(df_eda, target_col)
eda_pairplot(df_eda, target_col, problem_type)
eda_class_imbalance(df_eda, target_col, problem_type)


## Step 3: Preprocessing

Pipeline steps:
1. **Data cleaning** — drop unusable columns, remove duplicates
2. **Missing values** — drop columns with >50% missing; median (numeric) / mode (categorical) imputation
3. **Outliers** — IQR winsorization (cap, do not remove rows)
4. **Encoding** — One-Hot (≤5 unique) / Label (>5 unique)
5. **Feature selection** — remove features with correlation >0.95
6. **Train/test split** — 80/20 stratified for classification
7. **Scaling** — StandardScaler fit on training data only
8. **SMOTE** — applied on training data only (classification)


In [ ]:
def drop_sibling_label_columns(
    df: pd.DataFrame,
    target_col: str,
    sibling_cols: List[str],
) -> pd.DataFrame:
    """
    Remove sibling label columns from features when predicting one toxic label.
    Example: if target is 'toxic', drop severe_toxic, obscene, threat, insult, identity_hate.
    Prevents data leakage — those columns are other outputs, not real inputs.
    """
    to_drop = [c for c in sibling_cols if c in df.columns and c != target_col]
    if to_drop:
        print(f"Dropping sibling label columns from features: {to_drop}")
        PIPELINE_LOG["preprocessing"].append(
            f"Dropped sibling labels (leakage prevention): {to_drop}"
        )
        return df.drop(columns=to_drop)
    print("No sibling label columns to drop for this target.")
    return df


def maybe_subsample(df: pd.DataFrame, target: str, max_rows: Optional[int]) -> pd.DataFrame:
    """Subsample large datasets for feasible training in Colab."""
    if max_rows is None or len(df) <= max_rows:
        return df
    print(f"Subsampling {len(df):,} rows → {max_rows:,} for training efficiency (set MAX_TRAINING_ROWS=None to disable)")
    if df[target].nunique() <= 20:
        return df.groupby(df[target], group_keys=False).apply(
            lambda g: g.sample(
                min(len(g), max(1, int(max_rows * len(g) / len(df)))),
                random_state=RANDOM_STATE,
            )
        ).reset_index(drop=True)
    return df.sample(max_rows, random_state=RANDOM_STATE).reset_index(drop=True)


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate rows and log count removed."""
    before = len(df)
    df_clean = df.drop_duplicates().reset_index(drop=True)
    removed = before - len(df_clean)
    print(f"Duplicates removed: {removed}")
    PIPELINE_LOG["preprocessing"].append(f"Removed {removed} duplicate row(s)")
    return df_clean


def handle_missing_values(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """
    Drop columns with >50% missing (except target).
    Impute numeric with median, categorical with mode.
    """
    df_imp = df.copy()
    n_rows = len(df_imp)

    # Drop high-missing columns
    drop_cols = []
    for col in df_imp.columns:
        if col == target:
            continue
        miss_rate = df_imp[col].isnull().mean()
        if miss_rate > MISSING_DROP_THRESHOLD:
            drop_cols.append(col)
    if drop_cols:
        df_imp = df_imp.drop(columns=drop_cols)
        print(f"Dropped columns with >{MISSING_DROP_THRESHOLD*100:.0f}% missing: {drop_cols}")
        PIPELINE_LOG["preprocessing"].append(f"Dropped high-missing columns: {drop_cols}")

    # Impute remaining missing values
    for col in df_imp.columns:
        if col == target:
            continue
        if df_imp[col].dtype in [np.number]:
            median_val = df_imp[col].median()
            n_miss = df_imp[col].isnull().sum()
            if n_miss > 0:
                df_imp[col] = df_imp[col].fillna(median_val)
                print(f"  Median imputation: {col} ({n_miss} values)")
        else:
            mode_val = df_imp[col].mode(dropna=True)
            if len(mode_val) > 0 and df_imp[col].isnull().any():
                df_imp[col] = df_imp[col].fillna(mode_val.iloc[0])
                print(f"  Mode imputation: {col}")

    # Drop rows with missing target
    df_imp = df_imp.dropna(subset=[target]).reset_index(drop=True)
    PIPELINE_LOG["preprocessing"].append("Median/mode imputation applied; target NaNs dropped")
    return df_imp


def winsorize_outliers_iqr(df: pd.DataFrame, target: str) -> pd.DataFrame:
    """Cap numeric outliers using IQR-based winsorization. Rows are NOT removed."""
    df_cap = df.copy()
    num_cols = [c for c in df_cap.select_dtypes(include=[np.number]).columns if c != target]
    capped_count = 0

    for col in num_cols:
        q1 = df_cap[col].quantile(0.25)
        q3 = df_cap[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        before = df_cap[col].copy()
        df_cap[col] = df_cap[col].clip(lower=lower, upper=upper)
        capped_count += int((before != df_cap[col]).sum())

    print(f"Outlier capping (IQR winsorization): {capped_count} value(s) capped across numeric features")
    PIPELINE_LOG["preprocessing"].append(f"IQR winsorization capped {capped_count} outlier value(s)")
    return df_cap


def encode_features(df: pd.DataFrame, target: str) -> Tuple[pd.DataFrame, Dict[str, LabelEncoder]]:
    """
    One-Hot Encoding for categorical features with <=5 unique values.
    Label Encoding for categorical features with >5 unique values.
    """
    df_enc = df.copy()
    label_encoders: Dict[str, LabelEncoder] = {}
    feature_cols = [c for c in df_enc.columns if c != target]

    one_hot_cols = []
    label_cols = []

    for col in feature_cols:
        if df_enc[col].dtype in [np.number]:
            continue
        nunique = df_enc[col].nunique(dropna=True)
        if nunique <= 5:
            one_hot_cols.append(col)
        else:
            label_cols.append(col)

    # One-hot encode low-cardinality categoricals
    if one_hot_cols:
        df_enc = pd.get_dummies(df_enc, columns=one_hot_cols, drop_first=True)
        print(f"One-Hot Encoded (<=5 unique): {one_hot_cols}")

    # Label encode higher-cardinality categoricals
    for col in label_cols:
        le = LabelEncoder()
        df_enc[col] = le.fit_transform(df_enc[col].astype(str))
        label_encoders[col] = le
        print(f"Label Encoded (>5 unique): {col}")

    PIPELINE_LOG["preprocessing"].append(
        f"Encoding: One-Hot={one_hot_cols}, Label={label_cols}"
    )
    return df_enc, label_encoders


def encode_target(y: pd.Series, problem: str) -> Tuple[np.ndarray, Optional[LabelEncoder]]:
    """Label-encode classification targets if non-numeric."""
    if problem != "classification":
        return pd.to_numeric(y, errors="coerce").values, None
    if y.dtype in [np.number] and y.nunique() <= 20:
        return y.values.astype(int) if np.allclose(y.dropna(), y.dropna().astype(int)) else y.values, None
    le = LabelEncoder()
    encoded = le.fit_transform(y.astype(str))
    return encoded, le


def remove_highly_correlated_features(X: pd.DataFrame, threshold: float = CORR_THRESHOLD) -> pd.DataFrame:
    """Remove one feature from each pair with |correlation| > threshold."""
    if X.shape[1] < 2:
        return X
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    if to_drop:
        print(f"Dropped highly correlated features (|r|>{threshold}): {to_drop}")
        PIPELINE_LOG["preprocessing"].append(f"Dropped correlated features: {to_drop}")
        return X.drop(columns=to_drop)
    print("No highly correlated features to drop.")
    return X


def prepare_train_test_data(
    df: pd.DataFrame,
    target: str,
    problem: str,
) -> Dict[str, Any]:
    """Full preprocessing through train/test split, scaling, and SMOTE."""
    # Separate features and target
    y_series = df[target]
    X_df = df.drop(columns=[target])

    # Ensure all features are numeric after encoding
    X_df = X_df.select_dtypes(include=[np.number])
    if X_df.shape[1] == 0:
        raise ValueError("No numeric features available after preprocessing.")

    # Feature selection by correlation
    X_df = remove_highly_correlated_features(X_df)

    y, target_encoder = encode_target(y_series, problem)

    # Train/test split
    stratify = y if problem == "classification" and len(np.unique(y)) > 1 else None
    X_train, X_test, y_train, y_test = train_test_split(
        X_df.values,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify,
    )

    feature_names = list(X_df.columns)
    print(f"Train size: {X_train.shape} | Test size: {X_test.shape}")

    # Scaling — fit ONLY on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print("StandardScaler applied (fit on train only)")
    PIPELINE_LOG["preprocessing"].append("StandardScaler fit on training data only")

    # SMOTE on training data only (classification)
    smote_applied = False
    if problem == "classification":
        class_counts = np.bincount(y_train.astype(int))
        if len(class_counts) > 1 and class_counts.min() / class_counts.max() < 0.8:
            k_neighbors = min(5, class_counts.min() - 1)
            if k_neighbors >= 1:
                smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=k_neighbors)
                X_train_scaled, y_train = smote.fit_resample(X_train_scaled, y_train)
                smote_applied = True
                print(f"SMOTE applied on training data. New train size: {X_train_scaled.shape}")
                PIPELINE_LOG["preprocessing"].append(f"SMOTE applied (train size: {X_train_scaled.shape[0]})")
        else:
            print("SMOTE skipped — classes are relatively balanced or insufficient minority samples")
    else:
        print("SMOTE skipped — regression task")

    return {
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "feature_names": feature_names,
        "scaler": scaler,
        "target_encoder": target_encoder,
        "smote_applied": smote_applied,
    }


In [ ]:
# --- Execute Step 3: Preprocessing ---
df_clean = df_raw.copy()
df_clean = drop_unusable_columns(df_clean, target_col)
df_clean = remove_duplicates(df_clean)
df_clean = handle_missing_values(df_clean, target_col)
df_clean = winsorize_outliers_iqr(df_clean, target_col)
df_clean, label_encoders = encode_features(df_clean, target_col)
df_clean = maybe_subsample(df_clean, target_col, MAX_TRAINING_ROWS)

print("\nShape after preprocessing (before split):", df_clean.shape)

data = prepare_train_test_data(df_clean, target_col, problem_type)
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
feature_names = data["feature_names"]
target_encoder = data["target_encoder"]

print(f"\nFinal feature count: {len(feature_names)}")
print(f"Features: {feature_names[:15]}{'...' if len(feature_names) > 15 else ''}")


## Step 4: Model Training

**Ensemble methods used (only these five):**
1. Random Forest
2. Bagging
3. Gradient Boosting
4. Voting Classifier / Regressor
5. Stacking Classifier / Regressor

Hyperparameter tuning via `RandomizedSearchCV` (n_iter=20, cv=5, n_jobs=-1, random_state=42).

Voting and Stacking models are built using the three tuned base estimators.


In [ ]:
def get_scoring_metric(problem: str) -> str:
    """Primary scoring metric for RandomizedSearchCV."""
    return "f1_weighted" if problem == "classification" else "neg_mean_squared_error"


def get_param_distributions(problem: str) -> Dict[str, Dict]:
    """Hyperparameter search spaces for Random Forest, Bagging, and Gradient Boosting."""
    if problem == "classification":
        return {
            "Random Forest": {
                "n_estimators": randint(50, 300),
                "max_depth": [None] + list(randint(3, 20).rvs(5)),
                "min_samples_split": randint(2, 20),
                "min_samples_leaf": randint(1, 10),
                "max_features": ["sqrt", "log2", None],
            },
            "Bagging": {
                "n_estimators": randint(10, 150),
                "max_samples": uniform(0.5, 0.5),
                "max_features": uniform(0.5, 0.5),
            },
            "Gradient Boosting": {
                "n_estimators": randint(50, 300),
                "learning_rate": uniform(0.01, 0.29),
                "max_depth": randint(2, 10),
                "min_samples_split": randint(2, 20),
                "min_samples_leaf": randint(1, 10),
                "subsample": uniform(0.6, 0.4),
            },
        }
    return {
        "Random Forest": {
            "n_estimators": randint(50, 300),
            "max_depth": [None] + list(randint(3, 20).rvs(5)),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "max_features": ["sqrt", "log2", None],
        },
        "Bagging": {
            "n_estimators": randint(10, 150),
            "max_samples": uniform(0.5, 0.5),
            "max_features": uniform(0.5, 0.5),
        },
        "Gradient Boosting": {
            "n_estimators": randint(50, 300),
            "learning_rate": uniform(0.01, 0.29),
            "max_depth": randint(2, 10),
            "min_samples_split": randint(2, 20),
            "min_samples_leaf": randint(1, 10),
            "subsample": uniform(0.6, 0.4),
        },
    }


def build_base_estimators(problem: str):
    """Create untuned base estimators for RF, Bagging, and Gradient Boosting."""
    if problem == "classification":
        return {
            "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE),
            "Bagging": BaggingClassifier(
                estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
                random_state=RANDOM_STATE,
            ),
            "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
        }
    return {
        "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE),
        "Bagging": BaggingRegressor(
            estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
            random_state=RANDOM_STATE,
        ),
        "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }


def tune_models(
    X_train: np.ndarray,
    y_train: np.ndarray,
    problem: str,
) -> Dict[str, Any]:
    """
    Tune Random Forest, Bagging, and Gradient Boosting with RandomizedSearchCV.
    Returns dict of fitted best estimators and their best parameters.
    """
    scoring = get_scoring_metric(problem)
    param_grids = get_param_distributions(problem)
    base_estimators = build_base_estimators(problem)

    tuned_models = {}
    best_params_log = {}

    for name, estimator in base_estimators.items():
        print("\n" + "=" * 60)
        print(f"Tuning: {name}")
        print("=" * 60)

        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_grids[name],
            n_iter=N_ITER_SEARCH,
            cv=CV_FOLDS,
            scoring=scoring,
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        best_params_log[name] = search.best_params_
        print(f"Best score ({scoring}): {search.best_score_:.4f}")
        print(f"Best params: {search.best_params_}")

    PIPELINE_LOG["hyperparameters"] = best_params_log
    return tuned_models


def build_ensemble_models(tuned_models: Dict[str, Any], problem: str) -> Dict[str, Any]:
    """Build Voting and Stacking ensembles from tuned base estimators."""
    estimators = [(name.replace(" ", "_").lower(), model) for name, model in tuned_models.items()]

    if problem == "classification":
        voting = VotingClassifier(estimators=estimators, voting="soft", n_jobs=-1)
        stacking = StackingClassifier(
            estimators=estimators,
            final_estimator=RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
            cv=CV_FOLDS,
            n_jobs=-1,
            passthrough=False,
        )
    else:
        voting = VotingRegressor(estimators=estimators, n_jobs=-1)
        stacking = StackingRegressor(
            estimators=estimators,
            final_estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
            cv=CV_FOLDS,
            n_jobs=-1,
            passthrough=False,
        )

    return {"Voting": voting, "Stacking": stacking}


In [ ]:
# --- Execute Step 4: Model Training ---
print("Starting hyperparameter tuning (this may take several minutes)...")
tuned_models = tune_models(X_train, y_train, problem_type)

# Train tuned base models (already fitted during search)
trained_models = {name: model for name, model in tuned_models.items()}

# Build and train ensemble models
ensemble_models = build_ensemble_models(tuned_models, problem_type)
for name, model in ensemble_models.items():
    print(f"\nTraining {name} ensemble...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("\nAll models trained successfully!")
print("Models:", list(trained_models.keys()))


## Step 5: Evaluation

**Classification metrics:** Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix, ROC Curve, Classification Report

**Regression metrics:** MAE, MSE, RMSE, R²


In [ ]:
def evaluate_classification(model, X_test, y_test, model_name: str) -> Dict[str, float]:
    """Compute classification metrics and display diagnostic plots."""
    y_pred = model.predict(X_test)

    metrics = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

    # ROC-AUC (binary and multiclass)
    try:
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test)
            n_classes = len(np.unique(y_test))
            if n_classes == 2:
                metrics["ROC-AUC"] = roc_auc_score(y_test, y_proba[:, 1])
            else:
                metrics["ROC-AUC"] = roc_auc_score(
                    y_test, y_proba, multi_class="ovr", average="weighted"
                )
        else:
            metrics["ROC-AUC"] = np.nan
    except Exception:
        metrics["ROC-AUC"] = np.nan

    print("\n" + "=" * 60)
    print(f"EVALUATION — {model_name}")
    print("=" * 60)
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}" if not np.isnan(v) else f"  {k}: N/A")

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    plt.show()

    # ROC Curve (binary classification)
    if hasattr(model, "predict_proba") and len(np.unique(y_test)) == 2:
        fig, ax = plt.subplots(figsize=(6, 5))
        RocCurveDisplay.from_predictions(y_test, model.predict_proba(X_test)[:, 1], ax=ax)
        ax.set_title(f"ROC Curve — {model_name}")
        plt.tight_layout()
        plt.show()

    # Classification Report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return metrics


def evaluate_regression(model, X_test, y_test, model_name: str) -> Dict[str, float]:
    """Compute regression metrics."""
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    metrics = {
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_test, y_pred),
    }

    print("\n" + "=" * 60)
    print(f"EVALUATION — {model_name}")
    print("=" * 60)
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

    # Residual plot
    residuals = y_test - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(y_pred, residuals, alpha=0.5)
    axes[0].axhline(0, color="red", linestyle="--")
    axes[0].set_xlabel("Predicted")
    axes[0].set_ylabel("Residuals")
    axes[0].set_title(f"Residual Plot — {model_name}")
    axes[1].scatter(y_test, y_pred, alpha=0.5)
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    axes[1].set_title("Actual vs Predicted")
    plt.tight_layout()
    plt.show()

    return metrics


def evaluate_all_models(models: Dict[str, Any], X_test, y_test, problem: str) -> pd.DataFrame:
    """Evaluate every trained model and return metrics DataFrame."""
    all_metrics = {}
    for name, model in models.items():
        if problem == "classification":
            all_metrics[name] = evaluate_classification(model, X_test, y_test, name)
        else:
            all_metrics[name] = evaluate_regression(model, X_test, y_test, name)
    return pd.DataFrame(all_metrics).T


In [ ]:
# --- Execute Step 5: Evaluation ---
metrics_df = evaluate_all_models(trained_models, X_test, y_test, problem_type)
PIPELINE_LOG["metrics"] = metrics_df.to_dict()
display(metrics_df.round(4))


## Step 6: Model Comparison

Models are ranked from best to worst. The best model is highlighted in the comparison table and bar chart.


In [ ]:
def rank_models(metrics_df: pd.DataFrame, problem: str) -> Tuple[pd.DataFrame, str]:
    """
    Rank models best to worst.
    Classification: primary metric = F1 (then Accuracy).
    Regression: primary metric = R2 (higher is better).
    """
    df = metrics_df.copy()

    if problem == "classification":
        primary = "F1" if "F1" in df.columns else "Accuracy"
        ascending = False
        secondary = "Accuracy" if "Accuracy" in df.columns else primary
    else:
        primary = "R2"
        ascending = False
        secondary = "RMSE" if "RMSE" in df.columns else primary
        # For RMSE, lower is better — sort separately if needed
        if primary not in df.columns:
            primary = "RMSE"
            ascending = True

    ranked = df.sort_values(by=[primary], ascending=ascending)
    best_model = ranked.index[0]

    ranked["Rank"] = range(1, len(ranked) + 1)
    cols = ["Rank"] + [c for c in ranked.columns if c != "Rank"]
    ranked = ranked[cols]

    print("=" * 60)
    print("MODEL RANKING (Best → Worst)")
    print("=" * 60)
    display(ranked.round(4))

    return ranked, best_model


def plot_model_comparison(metrics_df: pd.DataFrame, problem: str, best_model: str) -> None:
    """Bar chart comparing models; best model highlighted."""
    if problem == "classification":
        plot_metrics = [m for m in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"] if m in metrics_df.columns]
    else:
        plot_metrics = [m for m in ["MAE", "MSE", "RMSE", "R2"] if m in metrics_df.columns]

    n_metrics = len(plot_metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(4 * n_metrics, 5))
    if n_metrics == 1:
        axes = [axes]

    colors = ["#2ecc71" if idx == best_model else "#3498db" for idx in metrics_df.index]

    for ax, metric in zip(axes, plot_metrics):
        values = metrics_df[metric]
        bar_colors = ["#2ecc71" if m == best_model else "#3498db" for m in values.index]
        sns.barplot(x=values.index, y=values.values, palette=bar_colors, ax=ax, legend=False)
        ax.set_title(metric)
        ax.set_xlabel("")
        ax.set_ylabel(metric)
        ax.tick_params(axis="x", rotation=45)
        for label in ax.get_xticklabels():
            if label.get_text() == best_model:
                label.set_fontweight("bold")
                label.set_color("#27ae60")

    plt.suptitle(f"Model Comparison (Best: {best_model})", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# --- Execute Step 6: Model Comparison ---
ranked_df, best_model_name = rank_models(metrics_df, problem_type)
plot_model_comparison(metrics_df, problem_type, best_model_name)
PIPELINE_LOG["best_model"] = best_model_name

print(f"\n🏆 BEST MODEL: {best_model_name}")
if problem_type == "classification" and "Accuracy" in metrics_df.columns:
    print(f"   Accuracy: {metrics_df.loc[best_model_name, 'Accuracy']:.4f}")
elif problem_type == "regression" and "R2" in metrics_df.columns:
    print(f"   R²: {metrics_df.loc[best_model_name, 'R2']:.4f}")


## Step 7: Final Report

An automated text report is generated, displayed below, and saved as `report.txt`.


In [ ]:
def generate_report(
    pipeline_log: Dict[str, Any],
    metrics_df: pd.DataFrame,
    ranked_df: pd.DataFrame,
    best_model: str,
    problem: str,
) -> str:
    """Build a comprehensive text report from pipeline artifacts."""
    ds = pipeline_log.get("dataset_summary", {})
    lines = [
        "=" * 70,
        "MACHINE LEARNING PROJECT — FINAL REPORT",
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        "=" * 70,
        "",
        "1. DATASET SUMMARY",
        "-" * 40,
        f"  File: {ds.get('filename', 'N/A')}",
        f"  Rows: {ds.get('rows', 'N/A'):,}" if isinstance(ds.get('rows'), int) else f"  Rows: {ds.get('rows', 'N/A')}",
        f"  Columns: {ds.get('columns', 'N/A')}",
        f"  Target Column: {ds.get('target_column', 'N/A')}",
        f"  Problem Type: {ds.get('problem_type', 'N/A').upper()}",
        "",
        "2. EDA FINDINGS",
        "-" * 40,
    ]
    for finding in pipeline_log.get("eda_findings", []):
        lines.append(f"  • {finding}")
    if not pipeline_log.get("eda_findings"):
        lines.append("  • (No EDA findings logged)")

    lines.extend([
        "",
        "3. PREPROCESSING SUMMARY",
        "-" * 40,
    ])
    for step in pipeline_log.get("preprocessing", []):
        lines.append(f"  • {step}")

    lines.extend([
        "",
        "4. HYPERPARAMETERS (Best from RandomizedSearchCV)",
        "-" * 40,
    ])
    for model_name, params in pipeline_log.get("hyperparameters", {}).items():
        lines.append(f"  {model_name}:")
        for k, v in params.items():
            lines.append(f"    - {k}: {v}")

    lines.extend([
        "",
        "5. EVALUATION METRICS",
        "-" * 40,
        metrics_df.round(4).to_string(),
        "",
        "6. MODEL RANKING",
        "-" * 40,
        ranked_df.round(4).to_string(),
        "",
        "7. BEST MODEL",
        "-" * 40,
        f"  ★ {best_model}",
    ])

    if problem == "classification" and best_model in metrics_df.index:
        acc = metrics_df.loc[best_model, "Accuracy"]
        lines.append(f"  Accuracy: {acc:.4f}")
        pipeline_log["conclusions"].append(f"Best model '{best_model}' achieved {acc:.2%} accuracy.")
    elif problem == "regression" and best_model in metrics_df.index:
        r2 = metrics_df.loc[best_model, "R2"]
        lines.append(f"  R²: {r2:.4f}")
        pipeline_log["conclusions"].append(f"Best model '{best_model}' achieved R²={r2:.4f}.")

    lines.extend([
        "",
        "8. CONCLUSIONS",
        "-" * 40,
    ])
    conclusions = pipeline_log.get("conclusions", []) or [
        f"The {best_model} ensemble model performed best on the held-out test set.",
        "Preprocessing (imputation, outlier capping, encoding, scaling) improved data quality.",
        "Hyperparameter tuning via RandomizedSearchCV optimized base estimators for Voting/Stacking.",
    ]
    for c in conclusions:
        lines.append(f"  • {c}")

    lines.extend([
        "",
        "9. FUTURE IMPROVEMENTS",
        "-" * 40,
        "  • Try additional feature engineering (polynomial features, domain-specific transforms)",
        "  • Expand hyperparameter search space or use more iterations",
        "  • Apply cross-validation on the full pipeline with a sklearn Pipeline object",
        "  • Collect more data or address class imbalance with alternative techniques",
        "  • Perform SHAP/LIME analysis for model interpretability",
        "  • Deploy best model with joblib/pickle for production inference",
        "",
        "=" * 70,
        "END OF REPORT",
        "=" * 70,
    ])

    return "\n".join(lines)


# --- Execute Step 7: Generate and save report ---
report_text = generate_report(PIPELINE_LOG, metrics_df, ranked_df, best_model_name, problem_type)

print(report_text)

# Save report to file
with open("report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)

print("\n✅ Report saved to report.txt")

# Optional: download report in Colab
files.download("report.txt")


## Conclusion

This notebook completed the full supervised ML pipeline:

| Step | Description |
|------|-------------|
| 1 | Dataset Loading (CSV / XLSX / JSON) |
| 2 | Exploratory Data Analysis |
| 3 | Preprocessing (cleaning, imputation, outliers, encoding, scaling, SMOTE) |
| 4 | Ensemble Model Training (RF, Bagging, GB, Voting, Stacking) |
| 5 | Model Evaluation |
| 6 | Model Comparison & Best Model Selection |
| 7 | Automated Report (`report.txt`) |

### Iterative Improvement Workflow

After running this notebook in Colab, paste back:
- **Errors / stack traces** → we fix the affected cells
- **Warnings** → we suppress or resolve root causes
- **Metrics / accuracy** → we tune preprocessing and hyperparameters
- **Screenshots / plots** → we refine visualizations or data handling

**Goal:** error-free execution, correct evaluation, functioning ensembles, and **≥90% accuracy** when the dataset supports it.

---
*Re-run from Step 1 with a new dataset anytime. Set `TARGET_COLUMN` in the config cell to override auto-detection.*
